In [ ]:
from flask import Flask, render_template, request, redirect, url_for, jsonify
from datetime import date
import json
import os

app = Flask(__name__)

DATA_FILE = 'data.json'

def load_data():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, 'r') as f:
            return json.load(f)
    return {'students': [], 'attendance': {}}

def save_data(data):
    with open(DATA_FILE, 'w') as f:
        json.dump(data, f)

@app.route('/')
def index():
    data = load_data()
    return render_template('index.html', students=data['students'])

@app.route('/add_student', methods=['POST'])
def add_student():
    name = request.form.get('name', '').strip()
    data = load_data()
    if name and name not in data['students']:
        data['students'].append(name)
        save_data(data)
    return redirect(url_for('index'))

@app.route('/remove_student/<name>', methods=['POST'])
def remove_student(name):
    data = load_data()
    if name in data['students']:
        data['students'].remove(name)
        for day in data['attendance']:
            data['attendance'][day].pop(name, None)
        save_data(data)
    return redirect(url_for('index'))

@app.route('/attendance', methods=['GET', 'POST'])
def attendance():
    data = load_data()
    today = str(date.today())
    if request.method == 'POST':
        attendance_data = {}
        for student in data['students']:
            status = request.form.get(f'attendance_{student}', 'absent')
            attendance_data[student] = status
        if today not in data['attendance']:
            data['attendance'][today] = {}
        data['attendance'][today].update(attendance_data)
        save_data(data)
        return redirect(url_for('view_records'))
    current = data['attendance'].get(today, {})
    return render_template('attendance.html', students=data['students'], today=today, current=current)

@app.route('/records')
def view_records():
    data = load_data()
    return render_template('records.html', attendance=data['attendance'], students=data['students'])

@app.route('/api/students', methods=['GET'])
def api_students():
    data = load_data()
    return jsonify(data['students'])

@app.route('/api/attendance', methods=['GET'])
def api_attendance():
    data = load_data()
    return jsonify(data['attendance'])

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)

In [ ]:
import pytest
import json

def test_add_student(client):
    response = client.post('/add_student', data={'name': 'Alice'}, follow_redirects=True)
    assert response.status_code == 200
    api_response = client.get('/api/students')
    students = json.loads(api_response.data)
    assert 'Alice' in students

def test_mark_attendance(client):
    client.post('/add_student', data={'name': 'Bob'}, follow_redirects=True)
    response = client.post('/attendance', data={'attendance_Bob': 'present'}, follow_redirects=True)
    assert response.status_code == 200
    api_response = client.get('/api/attendance')
    attendance = json.loads(api_response.data)
    assert len(attendance) > 0
    for day_records in attendance.values():
        assert day_records.get('Bob') == 'present'

def test_get_records(client):
    client.post('/add_student', data={'name': 'Charlie'}, follow_redirects=True)
    client.post('/attendance', data={'attendance_Charlie': 'absent'}, follow_redirects=True)
    response = client.get('/records')
    assert response.status_code == 200
    assert b'Charlie' in response.data

def test_remove_student(client):
    client.post('/add_student', data={'name': 'Dave'}, follow_redirects=True)
    api_before = client.get('/api/students')
    assert b'Dave' in api_before.data
    client.post('/remove_student/Dave', follow_redirects=True)
    api_after = client.get('/api/students')
    students = json.loads(api_after.data)
    assert 'Dave' not in students

In [ ]:
import threading
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

BASE_URL = 'http://localhost:5001'

def test_home_page_loads(driver):
    driver.get(BASE_URL)
    assert 'Student Attendance' in driver.title
    heading = driver.find_element(By.TAG_NAME, 'h1')
    assert 'Student Attendance' in heading.text

def test_add_student_via_ui(driver):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 10)
    input_field = wait.until(EC.presence_of_element_located((By.NAME, 'name')))
    input_field.clear()
    input_field.send_keys('SeleniumStudent')
    submit_btn = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
    submit_btn.click()
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'student-item')))
    assert 'SeleniumStudent' in driver.page_source

def test_attendance_page_accessible(driver):
    driver.get(f'{BASE_URL}/attendance')
    assert 'attendance' in driver.current_url
    assert driver.find_element(By.TAG_NAME, 'h1')

In [ ]:
# ============================================================
# COMPLETE LINE-BY-LINE EXPLANATION OF THE ENTIRE PROJECT
# ============================================================

# ─── app.py ────────────────────────────────────────────────

# from flask import Flask, render_template, request, redirect, url_for, jsonify
# Imports Flask core class + helpers: render_template renders HTML,
# request handles incoming form/query data, redirect sends HTTP 302,
# url_for builds URLs from function names, jsonify returns JSON responses.

# from datetime import date
# Imports the date class to get today's date for attendance records.

# import json
# Imports the built-in JSON module to read and write the data file.

# import os
# Imports the os module to check if the data file exists on disk.

# app = Flask(__name__)
# Creates the Flask application instance. __name__ tells Flask where
# to find templates and static files relative to this file.

# DATA_FILE = 'data.json'
# Constant for the JSON file path used as the database.

# def load_data():
# Helper function that reads all data from disk and returns it as a dict.

#     if os.path.exists(DATA_FILE):
#         Checks if data.json already exists before trying to open it.

#         with open(DATA_FILE, 'r') as f:
#             Opens the file in read mode using a context manager (auto-closes).

#             return json.load(f)
#             Parses the JSON file into a Python dict and returns it.

#     return {'students': [], 'attendance': {}}
#     Returns default empty structure when the file doesn't exist yet.
#     students is a list of names; attendance is a dict keyed by date.

# def save_data(data):
# Helper function that writes the current data dict back to disk as JSON.

#     with open(DATA_FILE, 'w') as f:
#         Opens file in write mode, creating it if it doesn't exist.

#         json.dump(data, f)
#         Serializes the Python dict to JSON and writes it to the file.

# @app.route('/')
# Registers the next function as the handler for GET requests to '/'.

# def index():
# Home page view — shows the list of students and the add-student form.

#     data = load_data()
#     Loads current data from disk.

#     return render_template('index.html', students=data['students'])
#     Renders the index.html template, passing the student list to it.
#     Jinja2 will replace {{ students }} in the HTML with the actual list.

# @app.route('/add_student', methods=['POST'])
# Only accepts POST requests — called when the HTML form is submitted.

# def add_student():
#     name = request.form.get('name', '').strip()
#     Gets the 'name' field from the submitted form. .strip() removes
#     leading/trailing whitespace. Defaults to empty string if missing.

#     data = load_data()
#     Loads current data so we can modify it.

#     if name and name not in data['students']:
#     Validates that name is not empty AND not already in the list
#     (prevents duplicates).

#         data['students'].append(name)
#         Adds the new student name to the in-memory list.

#         save_data(data)
#         Persists the updated list to disk.

#     return redirect(url_for('index'))
#     Sends a 302 redirect to '/' — the Post/Redirect/Get pattern
#     prevents duplicate form submissions on browser refresh.

# @app.route('/remove_student/<name>', methods=['POST'])
# Route with a dynamic segment <name> in the URL path.
# Flask passes the captured value as the 'name' function argument.

# def remove_student(name):
#     data = load_data()
#     if name in data['students']:
#         Checks that the student exists before trying to remove.

#         data['students'].remove(name)
#         Removes the first occurrence of name from the list.

#         for day in data['attendance']:
#             data['attendance'][day].pop(name, None)
#         Also removes this student's records from all attendance dates.
#         .pop(name, None) removes the key if present, does nothing if not.

#         save_data(data)
#         Saves the cleaned data to disk.

#     return redirect(url_for('index'))
#     Redirects back to the home page.

# @app.route('/attendance', methods=['GET', 'POST'])
# Handles both GET (show the form) and POST (save submitted form).

# def attendance():
#     data = load_data()
#     today = str(date.today())
#     Gets today's date as a string like '2026-11-02' to use as the key.

#     if request.method == 'POST':
#         Checks if the form was submitted (POST) vs just viewing (GET).

#         attendance_data = {}
#         Creates an empty dict to collect this submission's data.

#         for student in data['students']:
#             status = request.form.get(f'attendance_{student}', 'absent')
#             For each student, reads their radio button value from the form.
#             The input name is 'attendance_Alice', 'attendance_Bob', etc.
#             Defaults to 'absent' if the field is somehow missing.

#             attendance_data[student] = status
#             Stores 'present' or 'absent' for this student.

#         if today not in data['attendance']:
#             data['attendance'][today] = {}
#         Creates the date key in attendance if it doesn't exist yet.

#         data['attendance'][today].update(attendance_data)
#         Merges today's new values into any existing record for today.

#         save_data(data)
#         Persists the attendance data.

#         return redirect(url_for('view_records'))
#         After saving, redirect to the records page.

#     current = data['attendance'].get(today, {})
#     For GET requests, fetch today's existing attendance (if any)
#     so the form can pre-fill the radio buttons.

#     return render_template('attendance.html', students=data['students'], today=today, current=current)
#     Renders the attendance form with students, date, and existing values.

# @app.route('/records')
# def view_records():
#     Renders the records page showing all attendance history.

# @app.route('/api/students', methods=['GET'])
# def api_students():
#     return jsonify(data['students'])
#     REST API endpoint: returns the student list as a JSON array.
#     jsonify() sets Content-Type to application/json automatically.

# @app.route('/api/attendance', methods=['GET'])
# def api_attendance():
#     return jsonify(data['attendance'])
#     REST API endpoint: returns all attendance records as a JSON object.

# if __name__ == '__main__':
#     app.run(debug=True, host='0.0.0.0', port=5000)
#     Only starts the development server when this file is run directly.
#     host='0.0.0.0' makes it reachable from outside the container.
#     debug=True enables auto-reload and the interactive debugger.

# ─── tests/conftest.py ─────────────────────────────────────

# import pytest
# Imports pytest framework — fixtures are a pytest concept.

# @pytest.fixture
# def app(tmp_path):
#     Pytest fixture that creates an isolated test instance of the app.
#     tmp_path is a built-in pytest fixture providing a unique temp dir.

#     flask_app.config['TESTING'] = True
#     Enables Flask's testing mode: disables error catching for cleaner errors.

#     app_module.DATA_FILE = test_data_file
#     Redirects the data file to a temp location so tests don't touch production data.

#     yield flask_app
#     Provides the app to the test, then resumes here for cleanup after the test.

#     app_module.DATA_FILE = original_data_file
#     Restores the original DATA_FILE path after each test.

# @pytest.fixture
# def client(app):
#     return app.test_client()
#     Creates a test client — a fake browser for sending HTTP requests in tests.

# ─── tests/test_app.py ─────────────────────────────────────

# def test_add_student(client):
#     response = client.post('/add_student', data={'name': 'Alice'}, follow_redirects=True)
#     Sends a POST to /add_student simulating the HTML form submission.
#     follow_redirects=True follows the 302 redirect automatically.

#     assert response.status_code == 200
#     Verifies the final page returned HTTP 200 OK.

#     api_response = client.get('/api/students')
#     students = json.loads(api_response.data)
#     Calls the JSON API and parses the response body to a Python list.

#     assert 'Alice' in students
#     Verifies Alice was actually saved.

# def test_mark_attendance(client):
#     First adds a student, then marks them present via POST,
#     then verifies via the API that the attendance record was saved correctly.

# def test_get_records(client):
#     Adds a student, marks them absent, then checks that GET /records
#     returns 200 and contains the student's name in the HTML.

# def test_remove_student(client):
#     Adds a student, verifies they appear in the API,
#     removes them via POST, then verifies they're gone from the API.

# ─── tests/test_selenium.py ────────────────────────────────

# @pytest.fixture(scope='module')
# def server(server):
#     Starts the Flask server in a daemon background thread.
#     scope='module' means the server starts once for the whole test module.
#     thread.daemon = True means it auto-exits when the main thread ends.
#     time.sleep(2) gives the server time to start up before tests run.

# @pytest.fixture(scope='module')
# def driver(server):
#     options.add_argument('--headless')
#     Runs Chrome in headless mode (no visible window) for CI/CD environments.

#     d = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
#     ChromeDriverManager auto-downloads the correct ChromeDriver version.
#     Creates a real Chrome browser instance controlled by Selenium.

#     yield d
#     d.quit()
#     Provides the driver to tests, then closes Chrome after all tests run.

# def test_home_page_loads(driver):
#     driver.get(BASE_URL)
#     Tells Chrome to navigate to the local Flask app URL.

#     assert 'Student Attendance' in driver.title
#     Reads the browser tab's title and verifies it matches.

#     heading = driver.find_element(By.TAG_NAME, 'h1')
#     Finds the first <h1> element in the DOM.

#     assert 'Student Attendance' in heading.text
#     Reads the h1's visible text and verifies it matches.

# def test_add_student_via_ui(driver):
#     wait = WebDriverWait(driver, 10)
#     Creates an explicit wait object — waits up to 10 seconds for elements.

#     input_field = wait.until(EC.presence_of_element_located((By.NAME, 'name')))
#     Waits until an element with name='name' appears in the DOM.

#     input_field.send_keys('SeleniumStudent')
#     Types text into the input field just like a real user would.

#     submit_btn.click()
#     Clicks the submit button to send the form.

#     assert 'SeleniumStudent' in driver.page_source
#     Verifies the newly added student name appears in the page HTML.

print('All code and explanations loaded successfully.')